# Approach 5 — image, indication, and fused classification control

This notebook compares logistic-regression classifiers on frozen ImageNet ResNet-18 image embeddings, indication-text TF-IDF, and their concatenation. Labels are the dataset's coarse `class` field (`normal` / `non-normal`), not clinician-adjudicated finding labels. Results are exploratory and must not be used for patient care.

## Runtime and splits

A GPU speeds up image embedding extraction; CPU is supported but slower. The default uses up to 512 training and 256 validation studies for a first run. Set `MAX_PER_SPLIT=None` for the full splits after the pipeline works. The test split is deliberately not evaluated in this cell.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "config.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the cxr_second_reader project folder in VS Code/Jupyter.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Local project folder:", REPO_ROOT)


In [ ]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import transforms
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from scipy.sparse import csr_matrix, hstack
from src.experiment_utils import study_rows, ImageTextRows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_PER_SPLIT = 512  # Set to None for all frontal studies.
BATCH_SIZE = 32
print("device:", DEVICE)

In [ ]:
def labeled_rows(split):
    frame = study_rows(split, None)
    frame["label"] = frame["class"].fillna("").astype(str).str.lower().map(
        lambda value: 0 if value == "normal" else (1 if value in {"non-normal", "abnormal"} else np.nan)
    )
    frame = frame.dropna(subset=["label"]).reset_index(drop=True)
    frame["label"] = frame["label"].astype(int)
    if frame["label"].nunique() != 2:
        raise ValueError(f"Split {split} must contain both normal and non-normal rows.")
    # Keep development subsets deterministic and ensure both classes are represented.
    if MAX_PER_SPLIT is not None and len(frame) > MAX_PER_SPLIT:
        per_class = max(1, MAX_PER_SPLIT // 2)
        parts = [group.sample(n=min(len(group), per_class), random_state=42)
                 for _, group in frame.groupby("label")]
        frame = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
    return frame

train_df, val_df = labeled_rows("train"), labeled_rows("validation")
weights = ResNet18_Weights.DEFAULT
transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
backbone = resnet18(weights=weights)
image_encoder = nn.Sequential(*list(backbone.children())[:-1]).to(DEVICE).eval()
for parameter in image_encoder.parameters(): parameter.requires_grad = False

def image_embeddings(frame):
    loader = DataLoader(ImageTextRows(frame, transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    blocks=[]
    with torch.inference_mode():
        for batch in loader:
            blocks.append(image_encoder(batch["image"].to(DEVICE)).flatten(1).cpu().numpy())
    return np.concatenate(blocks).astype(np.float32)

X_image_train = image_embeddings(train_df)
X_image_val = image_embeddings(val_df)
y_train = train_df["label"].to_numpy(); y_val = val_df["label"].to_numpy()
image_scaler = StandardScaler()
X_image_train_scaled = image_scaler.fit_transform(X_image_train)
X_image_val_scaled = image_scaler.transform(X_image_val)
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=3000)
train_indication = train_df["indication"].fillna("").replace("", "no indication provided")
val_indication = val_df["indication"].fillna("").replace("", "no indication provided")
X_text_train = vectorizer.fit_transform(train_indication)
X_text_val = vectorizer.transform(val_indication)
X_fused_train = hstack([csr_matrix(X_image_train_scaled), X_text_train]).tocsr()
X_fused_val = hstack([csr_matrix(X_image_val_scaled), X_text_val]).tocsr()

In [ ]:
def fit_and_score(X_train, X_val, name):
    classifier = LogisticRegression(max_iter=1000, class_weight="balanced")
    classifier.fit(X_train, y_train)
    probability = classifier.predict_proba(X_val)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    return classifier, {
        "model": name,
        "accuracy": float(accuracy_score(y_val, prediction)),
        "balanced_accuracy": float(balanced_accuracy_score(y_val, prediction)),
        "f1_non_normal": float(f1_score(y_val, prediction, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_val, probability)) if len(np.unique(y_val)) == 2 else None,
        "validation_studies": int(len(y_val)),
    }

image_clf, image_metrics = fit_and_score(X_image_train_scaled, X_image_val_scaled, "image_only")
text_clf, text_metrics = fit_and_score(X_text_train, X_text_val, "indication_only")
fusion_clf, fusion_metrics = fit_and_score(X_fused_train, X_fused_val, "image_plus_indication")
metrics = [image_metrics, text_metrics, fusion_metrics]
display(pd.DataFrame(metrics))
OUT = Path("outputs/06_fusion_classifier"); OUT.mkdir(parents=True, exist_ok=True)
import joblib
joblib.dump({"image_classifier": image_clf, "text_classifier": text_clf,
             "fusion_classifier": fusion_clf, "image_scaler": image_scaler,
             "vectorizer": vectorizer, "backbone": "resnet18", "labels": ["normal", "non-normal"]},
            OUT / "classifier_bundle.joblib")
(OUT / "validation_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Saved validation-only classifiers to", OUT.resolve())

## Final test evaluation

After selecting a model using validation only, evaluate the test split once. Reuse the exact saved image scaler, text vectorizer, classifier, and ResNet preprocessing. Report class balance, balanced accuracy, F1, and ROC-AUC; do not interpret these coarse weak labels as diagnostic performance.

## Final held-out test evaluation
Run once after all model and threshold choices are frozen on validation. This cell writes per-study predictions and metrics to the model output folder.


In [ ]:
# Final held-out test: reuse validation-selected preprocessing and classifiers exactly.
test_df = labeled_rows("test")
X_image_test = image_embeddings(test_df)
X_image_test_scaled = image_scaler.transform(X_image_test)
test_indication = test_df["indication"].fillna("").replace("", "no indication provided")
X_text_test = vectorizer.transform(test_indication)
X_fused_test = hstack([csr_matrix(X_image_test_scaled), X_text_test]).tocsr()
y_test = test_df["label"].to_numpy()
results = []
prediction_rows = pd.DataFrame({"uid": test_df["uid"].astype(str), "label": y_test})
for name, classifier, features in [
    ("image_only", image_clf, X_image_test_scaled),
    ("indication_only", text_clf, X_text_test),
    ("image_plus_indication", fusion_clf, X_fused_test),
]:
    probability = classifier.predict_proba(features)[:, 1]
    predicted = (probability >= 0.5).astype(int)
    results.append({
        "model": name, "n": int(len(y_test)),
        "normal_n": int((y_test == 0).sum()), "non_normal_n": int((y_test == 1).sum()),
        "accuracy": float(accuracy_score(y_test, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, predicted)),
        "f1_non_normal": float(f1_score(y_test, predicted, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, probability)) if len(np.unique(y_test)) == 2 else None,
    })
    prediction_rows[f"{name}_prob_non_normal"] = probability
    prediction_rows[f"{name}_prediction"] = predicted
OUT.mkdir(parents=True, exist_ok=True)
prediction_rows.to_csv(OUT / "test_predictions.csv", index=False)
(OUT / "test_metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
display(pd.DataFrame(results))
print("These are coarse dataset labels, not diagnostic performance. Review labels and outputs clinically before any use.")
